# Reasoning Model Fine-Tuning with Unsloth and GRPO

In this session, we step away from orchestrating models and change the weights themselves. Using Unsloth's implementation of Group Relative Policy Optimization (GRPO) — the reinforcement learning algorithm behind DeepSeek-R1 — we will teach `meta-llama/Llama-3.2-3B-Instruct` to produce structured, step-by-step reasoning on grade-school math problems (GSM8K).

This is not a 1-to-1 reproduction of the DeepSeek-R1 paper, but it exercises the same core loop:

```text
prompt -> sample a group of 8 completions -> score each with reward functions
       -> compute group-relative advantage -> policy update (with KL penalty) -> repeat
```

Because each completion is scored relative to the *group average*, GRPO needs no separate value network (critic) — a big part of why it is practical at small scale. And unlike supervised fine-tuning, we never show the model *how* to reason: we only reward it when the reasoning format and final answer are right, and let it discover the rest.

We train a **16-bit LoRA adapter** (not QLoRA): GRPO spends most of its time generating completions, and vLLM's fast-generation path is fastest and most stable on 16-bit weights.

> **Hardware requirements:** this notebook trains locally and needs an NVIDIA GPU with compute capability 8.0+ (Ampere or newer — recent vLLM releases dropped T4/Turing support), 16 GB+ VRAM, on Linux or WSL2. See the session README for prerequisites and out-of-memory knobs.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain the GRPO training loop and how it differs from SFT and PPO-style RLHF.
- Load a base model with Unsloth and attach LoRA adapters for parameter-efficient training.
- Explain the trade-off between 16-bit LoRA and 4-bit QLoRA.
- Prepare a verifiable-answer dataset (GSM8K) for reward-based training.
- Design stacked reward functions that shape both format and correctness.
- Configure and run TRL's `GRPOTrainer` and interpret its reward logs.
- Compare base and fine-tuned behavior, and save and load the trained LoRA adapter.

# Colab-Specific Notebook Information 

- You will need a Google Colab Pro account. If you do not already have one, set it up now.
- Colab uses `%pip install` instead of `uv sync`
- This notebook has been successfully run using:
  - The ungated `unsloth/Llama-3.2-3B-Instruct` mirror rather than the HuggingFace licensed version.
  - the GRPOConfig `report_to` parameter set to "none" instead of "wandb" _(NOTE: The wandb library is installed  via `%pip install` if you want to use it.)_
- The notebook was run using an `L4/A100` runtime. There is a configuration check in the Environment Setup section (Task 1) of the notebook.

## Table of Contents

- **Breakout Room #1: Model Loading and LoRA**
  - Task 1: Environment Setup
  - Task 2: Load the Base Model
  - Task 3: Attach LoRA Adapters
  - Question #1, Question #2, and Question #3
- **Breakout Room #2: GRPO Training on GSM8K**
  - Task 4: Prepare the GSM8K Dataset
  - Task 5: Define Reward Functions
  - Task 6: Configure GRPO
  - Question #4
  - Task 7: Train with GRPOTrainer
  - Task 8: Compare Before and After
  - Task 9: Save and Load the LoRA

---
# Breakout Room #1
## Model Loading and LoRA

We load the base model with Unsloth's vLLM-backed fast-inference path, then attach LoRA adapters so training touches only a small fraction of the weights.

## Task 1: Environment Setup

_NOTE: If you want to use the gated repository version of `meta-llama/Llama-3.2-3B-Instruct` you will need to:_
- Log in or sign up on [huggingface.co](https://huggingface.co); and then
- Accept the license on its [Hugging Face model page](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
- Wait until you receive an email stating your access has been granted by the repository authors
- Uncomment the code cell that allows you to log into HuggingFace

### Imports

This notebook uses:

- Unsloth `FastLanguageModel` for memory-efficient model loading and LoRA
- vLLM (via Unsloth's `fast_inference` path) for high-throughput generation during training
- TRL `GRPOConfig` and `GRPOTrainer` for the GRPO training loop
- `datasets` for loading GSM8K

_**IMPORTANT: The following code cell will prompt you to restart the session. You must do this before continuing. There are three ways to do this:**_
1. Via the modal that pops up
2. Via the `RESTART SESSION` button iin the cell's output log
3. Using the `Runtime -> Restart session` menu option.

In [ ]:
!nvidia-smi
import torch
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

%pip install -U \
  "unsloth==2026.7.4" \
  "vllm==0.15.1" \
  "trl==0.22.2" \
  "transformers==4.56.2" \
  "datasets>=3.4.1,<4.4.0"
%pip install "wandb>=0.19.0"

> NOTE: `UNSLOTH_VLLM_STANDBY` must be set **before** importing Unsloth — it lets training and vLLM share GPU memory instead of splitting it, freeing roughly 30% more context headroom.

In [ ]:
import os

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # must be set before importing unsloth

The following code will print out something similar to this if you have the correct runtime configured:
```
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2.9.1+cu128 NVIDIA L4
0.15.1 0.22.2 4.56.2
```

In [ ]:
import torch
from unsloth import FastLanguageModel
import vllm, trl, transformers, datasets

print(torch.__version__, torch.cuda.get_device_name(0))
print(vllm.__version__, trl.__version__, transformers.__version__)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Colab/Unsloth set expandable_segments; incompatible with vLLM standby
for key in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_ALLOC_CONF"):
    os.environ.pop(key, None)

assert torch.cuda.is_available(), "This notebook requires an NVIDIA GPU."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB | Compute: {gpu.major}.{gpu.minor}")

Uncomment an then run the following cell to use the HuggingFace model repo. You will also need to swap the `model_name` parameter in Task 2.

In [ ]:
#import getpass
#
#from huggingface_hub import get_token, login
#
#if get_token() is None:
#    login(token=getpass.getpass("Hugging Face token (with the Llama 3.2 license accepted): "))

## Task 2: Load the Base Model

Unsloth wraps Hugging Face model loading with `FastLanguageModel.from_pretrained`. The arguments that matter here:

- `max_seq_length = 2048` — the total budget for prompt + completion. Reasoning traces are long, so GRPO wants headroom; increase this if you train for longer traces.
- `load_in_4bit = False` — we deliberately train LoRA in **16-bit**. GRPO spends most of its wall-clock generating candidate completions, and vLLM's fast-generation path is fastest and most stable on 16-bit weights. Set this to `True` (QLoRA) only if you need to squeeze under ~8 GB of VRAM.
- `fast_inference = True` — runs the vLLM engine inside Unsloth so group sampling doesn't crawl.
- `max_lora_rank = 64` — must be at least the LoRA rank we pick in Task 3.
- `gpu_memory_utilization = 0.7` — the fraction of VRAM vLLM may claim for weights + KV cache. `0.7` fits a 16 GB card; raise toward `0.9` on 24 GB.

### A quick word on quantization (and why we skip it here)

Quantization discretizes weights from a representation that holds more information into one that holds less. The [QLoRA paper](https://arxiv.org/abs/2305.14314) made 4-bit fine-tuning practical with two tricks: the **NF4** data type (near-optimal for normally distributed weights, applied block-wise with one scaling constant per 64-weight block) and **double quantization** (quantizing the quantization constants themselves, saving ~0.37 bits per parameter). That is the right tool when GPU memory is the binding constraint. Here, generation throughput is the binding constraint — so we keep the weights in 16-bit and let LoRA keep the *trainable* footprint small instead.

_NOTE: To use the HuggingFace model repo you need to swap the commenting of the `model_name` parameter in the following code block._

In [ ]:
max_seq_length = 2048  # can increase for longer reasoning traces
lora_rank = 64  # larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    # model_name = "meta-llama/Llama-3.2-3B-Instruct", 
    max_seq_length = max_seq_length,
    load_in_4bit = False,  # 16-bit LoRA — see the Task 2 notes
    fast_inference = True,  # enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7,  # raise toward 0.9 on a 24 GB card
)

## Task 3: Attach LoRA Adapters

Fine-tuning all 3B parameters is prohibitively expensive. [LoRA](https://arxiv.org/abs/2106.09685) — a PEFT (parameter-efficient fine-tuning) method — freezes the base weights and learns a low-rank update instead: for a weight matrix `W`, it trains two small matrices `A` and `B` of rank `r` such that the effective weight becomes `W + BA`. Only `A` and `B` receive gradients.

The knobs:

- `r = 64` — the rank of the decomposition. Higher rank = more trainable capacity (and more VRAM and time). Common values: 8, 16, 32, 64, 128.
- `lora_alpha = 64` — a scaling factor on the update; setting `alpha = r` is a common, stable default.
- `target_modules` — which weight matrices get adapters. The original LoRA paper adapted only attention weights; the QLoRA paper found adapting **all** linear layers works better, so we take the four attention projections plus the three MLP projections.
- `use_gradient_checkpointing = "unsloth"` — Unsloth's checkpointing variant, tuned for long-context RL training.
- `random_state = 3407` — reproducible adapter initialization.

The `print(model)` output below shows exactly where the `lora_A`/`lora_B` pairs were injected — you will need it for Question #2.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",  # enables long-context fine-tuning
    random_state = 3407,
)

print(model)

#### ❓ Question #1

The [QLoRA paper](https://arxiv.org/abs/2305.14314) introduces *double quantization*. In your own words: what gets quantized the second time, and roughly how much memory does it save per parameter? This notebook sets `load_in_4bit = False` and trains a 16-bit LoRA instead — why is 16-bit the better fit for GRPO with vLLM fast inference, and when would you still reach for QLoRA?

##### Answer:

_Write your answer here._

#### ❓ Question #2

![Transformer decoder diagram](https://i.imgur.com/N8y2crZ.png)

Using the `print(model)` output from Task 3, label the diagram with the matching layers from `meta-llama/Llama-3.2-3B-Instruct`'s architecture.

- EXAMPLE — Layer Norm:
  - `(input_layernorm): LlamaRMSNorm()`
  - `(post_attention_layernorm): LlamaRMSNorm()`
  - `(norm): LlamaRMSNorm()`
- Feed Forward:
- Masked Multi Self-Attention:
- Text & Position Embed:
- Text Prediction:

##### Answer:

_Write your answer here._

#### ❓ Question #3

What, in your own words, is LoRA doing?

##### Answer:

_Write your answer here._

## Breakout Room #1 Summary

- The base model loads once in 16-bit, with vLLM (`fast_inference = True`) ready for high-throughput group sampling.
- LoRA freezes the base weights and trains low-rank `A`/`B` updates on all seven projection matrices — a small fraction of the total parameters.
- 16-bit LoRA is the right default for GRPO; QLoRA (4-bit) trades generation speed for memory when VRAM is the constraint.

---
# Breakout Room #2
## GRPO Training on GSM8K

GRPO turns fine-tuning into a reward game. Each step:

1. **Group sampling** — for a single prompt, the policy generates a *group* of completions (8 here) instead of just one.
2. **Reward scoring** — each completion is scored by our reward functions.
3. **Group-based advantage** — each completion's reward is compared to the group average: above average means a positive advantage, below means negative.
4. **Policy update** — the policy is nudged toward positive-advantage outputs, with a KL penalty preventing drastic drift.
5. **Iterate** — the updated policy samples the next groups, and the loop repeats until reward converges.

The group baseline replaces the separate value network (critic) that PPO-style RLHF needs — and no human preference data or process reward model is involved. All we need is a dataset whose answers we can *verify*.

## Task 4: Prepare the GSM8K Dataset

Notice something odd about the data: it is just questions and final answers — no reasoning traces, no preference pairs. That is the point. We never show the model *how* to reason; we only need a way to check whether an answer is right so we can reward it.

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) marks each gold answer after a `####` delimiter, which makes verification a string comparison.

To make completions *checkable*, the system prompt forces a structure we can parse:

```text
<reasoning> ... </reasoning>
<answer> ... </answer>
```

`extract_xml_answer` pulls the model's answer out of that structure; `extract_hash_answer` pulls the gold answer out of GSM8K's `####` format.

> NOTE: This is not exactly the DeepSeek-R1 recipe — R1 adds a small SFT "cold start" stage to prime the model before RL. We skip straight to RL. The data prep and reward functions below build directly on [@willccbb's gist](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb), by way of Unsloth.

In [ ]:
import re

from datasets import Dataset, load_dataset

SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    """Return the text between the <answer> and </answer> tags."""
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    """Return the gold answer after GSM8K's '####' marker, or None if absent."""
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

def get_gsm8k_questions(split="train") -> Dataset:
    """Load GSM8K and map each item to a system+user prompt plus the extracted gold answer."""
    data = load_dataset("openai/gsm8k", "main")[split]
    data = data.map(lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x["question"]},
        ],
        "answer": extract_hash_answer(x["answer"]),
    })
    return data

dataset = get_gsm8k_questions()

In [ ]:
dataset[0]

## Task 5: Define Reward Functions

Here is the *magic* of the approach: instead of showing the model examples of good reasoning, we score its attempts with a stack of reward functions and let GRPO figure out the rest.

![Reward function stack](https://i.imgur.com/7Dp0qdt.png)

| Reward function | What it checks | Max reward |
|---|---|---|
| `correctness_reward_func` | the extracted answer equals the gold answer | 2.0 |
| `int_reward_func` | the extracted answer is a plain integer | 0.5 |
| `strict_format_reward_func` | exact `<reasoning>`/`<answer>` layout, newlines and all | 0.5 |
| `soft_format_reward_func` | the tags appear in the right order (lenient) | 0.5 |
| `xmlcount_reward_func` | partial credit per correctly placed tag, minus a trailing-junk penalty | ~0.5 |

Correctness dominates (2.0 vs 0.5), but the format rewards give the model a *gradient to climb* early on — a completion can earn partial credit for structure before it ever gets an answer right. These functions are fully customizable: they are how you steer what the model is incentivized to get good at.

In [ ]:
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """2.0 if the extracted <answer> matches the gold answer, else 0.0.

    Also prints the first question/response pair of each batch so you can watch
    completions evolve from rambling answers into structured reasoning.
    """
    responses = [completion[0]["content"] for completion in completions]
    q = prompts[0][-1]["content"]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print(
        "-" * 20,
        f"Question:\n{q}",
        f"\nGold answer:\n{answer[0]}",
        f"\nResponse:\n{responses[0]}",
        f"\nExtracted:\n{extracted_responses[0]}",
        sep="\n",
    )
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    """0.5 if the extracted answer is a plain integer, else 0.0."""
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

In [ ]:
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """0.5 if the completion matches the exact expected layout, newlines and all."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    # re.DOTALL lets .*? span newlines — without it, multi-line reasoning never matches
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """0.5 if <reasoning> and <answer> blocks appear in order, however loosely."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

In [ ]:
def count_xml(text) -> float:
    """Partial credit (0.125 each) per correctly placed tag; small penalty for trailing junk."""
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

In [ ]:
# Quick sanity check: soft accepts a single-line completion, strict wants each tag on its own line.
sample = [[{"content": "<reasoning>Wow, cool!</reasoning><answer>23</answer>"}]]
print("soft:  ", soft_format_reward_func(sample))
print("strict:", strict_format_reward_func(sample))

## Task 6: Configure GRPO

With training examples and reward functions in hand, all that is left is configuration. Two batch-geometry rules to understand first:

- `num_generations = 8` — the *group size* GRPO compares within. Decrease to 4 if you run out of memory (noisier advantages, less VRAM).
- TRL requires that `per_device_train_batch_size × gradient_accumulation_steps` be **divisible by `num_generations`** — the sampled groups must tile evenly into the effective batch. That is why we use `gradient_accumulation_steps = 8` here.

The rest are familiar fine-tuning knobs (`learning_rate`, `warmup_ratio`, `lr_scheduler_type` — see Question #4), plus:

- `max_prompt_length` / `max_completion_length` — we measure the longest tokenized prompt in the dataset and give completions everything that remains of `max_seq_length`.
- `optim = "adamw_8bit"` — 8-bit optimizer states shave a couple of GB of VRAM.
- `max_steps = 175` — enough to see the reward curve turn upward within a class session; for a full training run, train for an epoch or more.
- `report_to = "none"` — flip to `"wandb"` for the classic "line goes up and to the right" chart (install with `uv sync --extra wandb`, then `wandb login`).

In [ ]:
max_prompt_length = max(dataset.map(
    lambda x: {"tokens": tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
    batched=True,
).map(lambda x: {"length": len(x["tokens"])})["length"])

max_prompt_length = max_prompt_length + 1  # +1 just in case!
print(f"Longest prompt: {max_prompt_length} tokens")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate = 5e-6,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,  # batch_size x grad_accum must be divisible by num_generations
    num_generations = 8,  # group size; decrease to 4 if out of memory (and set grad_accum to 4)
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    max_steps = 175,
    save_steps = 25,
    max_grad_norm = 0.1,
    report_to = "none",  # flip to "wandb" for experiment tracking
    output_dir = "outputs",
)

#### ❓ Question #4

Describe what the following parameters are doing:

- `warmup_ratio`
- `learning_rate`
- `lr_scheduler_type`

> NOTE: Feel free to consult the [TrainingArguments documentation](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments) or other resources!

##### Answer:

_Write your answer here._

## Task 7: Train with GRPOTrainer

Unlike supervised fine-tuning, where you watch loss go *down*, here you watch reward go *up*. What to expect:

- Each logged step shows one column per reward function plus the combined `reward` — you can see the format rewards kick in before correctness does.
- The debug print in `correctness_reward_func` lets you watch raw completions evolve in real time.
- There is a famous "Aha!" moment: reward hovers near ~0 and then suddenly starts climbing. Expect little movement before step ~100–150, and budget roughly an hour at these settings. (Unsloth recommends 300+ steps for clearly strong results — 175 keeps this class-sized.)

Here is what the reward curve looked like on a previous run of this notebook — noisy at this scale, but unmistakably up and to the right:

![Example reward curve](https://i.imgur.com/7sBk5y2.png)

_Note: The next code cell can take 15-30 minutes to complete on Colab. Make sure you keep your system awake while it processes._

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

## Task 8: Compare Before and After

Time for the payoff. A subtle but important detail: vLLM is still holding the *frozen base weights* — LoRA never touched them. That means we can generate from the plain base model at any time by passing `lora_request = None`, and from our trained model by passing the saved adapter. Same GPU, same engine, two personalities.

First, the base model — no system prompt, no LoRA:

In [ ]:
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)

text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Calculate pi."}],
    tokenize = False,
    add_generation_prompt = True,
)

base_output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,  # frozen base weights only
)[0].outputs[0].text

print(base_output)

## Task 9: Save and Load the LoRA

`save_lora` writes only the adapter (the `A`/`B` matrices) — a few hundred megabytes at rank 64, versus multiple gigabytes for full weights. We then reload it as a `lora_request` and rerun the same prompt, this time with the reasoning system prompt the model was trained against:

In [ ]:
model.save_lora("grpo_saved_lora")

In [ ]:
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Calculate pi."},
    ],
    tokenize = False,
    add_generation_prompt = True,
)

trained_output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

print(trained_output)

## Breakout Room #2 Summary

- GSM8K needs no reasoning traces or preference data — verifiable final answers are enough for RL.
- Stacked reward functions shape behavior: cheap format rewards provide an early gradient, and correctness (2.0) dominates once the model starts solving problems.
- GRPO's group-relative advantage (a group of 8 here) replaces PPO's value network; TRL enforces that the effective batch tiles evenly into groups.
- Training shows a delayed "Aha!" — flat reward, then liftoff — rather than a smoothly descending loss.
- The trained artifact is just a LoRA adapter: swap it in and out of the same vLLM engine via `lora_request`.

Where to go next: train longer (300+ steps), try a larger base model, write your own reward functions, or export merged 16-bit weights for serving — the [Unsloth documentation](https://docs.unsloth.ai/) covers all of the above.